In [1]:
from pyspark.sql import SparkSession

In [ ]:
jars_path = '/home/ubuntu/work_dir/jars/'
jars = (
    'postgresql-42.7.13.jar',
    'hadoop-aws-3.5.0.jar',
    'analyticsaccelerator-s3-1.3.1.jar',
    'aws-java-sdk-bundle-2.55.16.jar',
    'delta-spark_2.13-4.4.1.jar'
)
spark_jars = ','.join(jars_path + jar for jar in jars)

In [3]:
spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .config('spark.eventLog.enabled','true')
    .config('spark.eventLog.dir','/tmp/spark-events')
    .config('spark.jars',spark_jars)
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://rustfs:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin") \
    .appName("notebook")
    .getOrCreate()
)

26/10/31 10:13:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
spark

In [5]:
# spark.range(1000).count()

In [6]:
df = spark.read.csv("s3a://raw/InstaCart/orders.csv", header=True)
df.show()

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
| 2539329|      1|   prior|           1|        2|               08|                  NULL|
| 2398795|      1|   prior|           2|        3|               07|                  15.0|
|  473747|      1|   prior|           3|        3|               12|                  21.0|
| 2254736|      1|   prior|           4|        4|               07|                  29.0|
|  431534|      1|   prior|           5|        4|               15|                  28.0|
| 3367565|      1|   prior|           6|        2|               07|                  19.0|
|  550135|      1|   prior|           7|        1|               09|                  20.0|
| 3108588|      1|   prior|           8|        1|               14|            

In [7]:
df.write.format('delta').save('s3a://raw/temp/orders')

AnalysisException: [DELTA_CONFIGURE_SPARK_SESSION_WITH_EXTENSION_AND_CATALOG] This Delta operation requires the SparkSession to be configured with the
DeltaSparkSessionExtension and the DeltaCatalog. Please set the necessary
configurations when creating the SparkSession as shown below.

  SparkSession.builder()
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    ...
    .getOrCreate()

If you are using spark-shell/pyspark/spark-submit, you can add the required configurations to the command as show below:
--conf spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog


In [5]:
df = spark.read.csv('data/InstaCart/aisles.csv',header=True)

In [6]:
df.head()

Row(aisle_id='1', aisle='prepared soups salads')

In [7]:
df.write \
.format('jdbc') \
.option("url", "jdbc:postgresql://instacart:5432/instacart") \
.option("dbtable", "staging_text.aisle") \
.option("user", "admin") \
.option("password", "admin") \
.option("driver", "org.postgresql.Driver") \
.mode("overwrite") \
.save()

In [6]:
spark.stop()